<a href="https://colab.research.google.com/github/AsimaZaheer/Stress-Detection/blob/main/integrate%20optimization%20murtaza%20V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# ============================================================
# WESAD STRESS DETECTION - OPTIMIZED VERSION
# 32 Hz + 3-Layer 1D CNN + Transformer + HRV
# LOSO (Leave-One-Subject-Out) Validation
# ============================================================
# Added optimizations:
# 1. Learnable positional encoding
# 2. HRV extraction from original 64-Hz BVP
# 3. AdamW + weight decay
# 4. ReduceLROnPlateau learning-rate scheduler
# 5. Early stopping
#
# Main experimental settings remain:
# 32 Hz CNN input, 60-s windows, 10-s step, 15-subject LOSO.
# ============================================================

import os
import pickle
import zipfile
import subprocess
import sys

import numpy as np
from scipy import signal
from scipy.signal import find_peaks

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. CONFIGURATION
# ============================================================

SUBJECTS = [
    "S2", "S3", "S4", "S5", "S6", "S7",
    "S8", "S9", "S10", "S11", "S13", "S14",
    "S15", "S16", "S17"
]

TARGET_SAMPLING_RATE = 32
BVP_NATIVE_RATE = 64
LABEL_NATIVE_RATE = 700

WINDOW_SIZE = 60
STEP_SIZE = 10

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3

# AdamW
WEIGHT_DECAY = 1e-4

# Learning-rate scheduler
LR_FACTOR = 0.5
LR_PATIENCE = 2
MIN_LEARNING_RATE = 1e-6

# Early stopping
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4

# Dropout values kept the same as the previous model
CNN_DROPOUT = 0.2
CLASSIFIER_DROPOUT = 0.3

# Transformer
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_DROPOUT = 0.2

# 60 sec * 32 Hz = 1920 samples.
# Three MaxPool1D(2): 1920 -> 960 -> 480 -> 240.
MAX_TRANSFORMER_LENGTH = (
    WINDOW_SIZE * TARGET_SAMPLING_RATE // 8
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. GOOGLE DRIVE DATASET STORAGE
# ============================================================

WESAD_ZIP_URL = (
    "https://drive.google.com/file/d/"
    "1MbfU2z4OnyevB0oX_yvHVue7Wv24KP7l"
    "/view?usp=sharing"
)

DRIVE_WESAD_DIR = "/content/drive/MyDrive/WESAD"
ZIP_PATH = os.path.join(DRIVE_WESAD_DIR, "WESAD.zip")
EXTRACT_PATH = DRIVE_WESAD_DIR

from google.colab import drive

print("=" * 70)
print("MOUNTING GOOGLE DRIVE")
print("=" * 70)
drive.mount("/content/drive")
os.makedirs(DRIVE_WESAD_DIR, exist_ok=True)
print("\nGoogle Drive mounted.")
print("WESAD storage location:")
print(DRIVE_WESAD_DIR)


# ============================================================
# 3. DOWNLOAD / EXTRACT DATASET
# ============================================================

def ensure_gdown():
    try:
        import gdown
        return gdown
    except ImportError:
        print("Installing gdown...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "gdown"
        ])
        import gdown
        return gdown


def find_subject_file(dataset_root, subject):
    target_name = f"{subject}.pkl"
    direct_path = os.path.join(dataset_root, subject, target_name)
    if os.path.isfile(direct_path):
        return direct_path

    for root, _, files in os.walk(dataset_root):
        if target_name in files:
            return os.path.join(root, target_name)
    return None


def download_wesad_dataset():
    print("=" * 70)
    print("WESAD DATASET SETUP")
    print("=" * 70)

    os.makedirs(DRIVE_WESAD_DIR, exist_ok=True)

    # If extracted data already exists, do nothing.
    existing_s2 = find_subject_file(EXTRACT_PATH, "S2")
    if existing_s2 is not None:
        print("\nWESAD dataset already exists in Google Drive.")
        print("No download required.")
        print("No extraction required.")
        dataset_root = os.path.dirname(os.path.dirname(existing_s2))
        print("\nS2.pkl found:")
        print(existing_s2)
        print("\nDataset root:")
        print(dataset_root)
        return dataset_root

    gdown = ensure_gdown()

    if not os.path.isfile(ZIP_PATH):
        print("\nWESAD ZIP not found in Google Drive.")
        print("Downloading WESAD ZIP...")
        gdown.download(WESAD_ZIP_URL, ZIP_PATH, quiet=False, fuzzy=True)
    else:
        print("\nWESAD ZIP already exists in Google Drive.")
        print(ZIP_PATH)

    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(
            f"Download failed. ZIP was not created:\n{ZIP_PATH}"
        )

    if not zipfile.is_zipfile(ZIP_PATH):
        raise RuntimeError(
            "Downloaded file is not a valid ZIP file. "
            "If the public source has a download quota problem, "
            "place a valid WESAD.zip manually in MyDrive/WESAD/."
        )

    print("\nExtracting WESAD ZIP to Google Drive...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_PATH)
    print("Extraction completed.")

    s2_file = find_subject_file(EXTRACT_PATH, "S2")
    if s2_file is None:
        raise FileNotFoundError(
            "S2.pkl was not found after extraction. "
            "Please check the WESAD ZIP structure."
        )

    dataset_root = os.path.dirname(os.path.dirname(s2_file))
    print("\nS2.pkl found:")
    print(s2_file)
    print("\nDataset root:")
    print(dataset_root)
    return dataset_root


# ============================================================
# 4. SIGNAL RESAMPLING
# ============================================================

def resample_signal(x, original_fs, target_fs):
    x = np.asarray(x, dtype=np.float32).squeeze()
    if x.ndim != 1:
        raise ValueError(f"Expected 1-D signal, got shape {x.shape}")
    if len(x) == 0:
        return x
    if original_fs == target_fs:
        return x.astype(np.float32)

    new_length = max(
        int(round(len(x) * target_fs / original_fs)), 1
    )
    return signal.resample(x, new_length).astype(np.float32)


# ============================================================
# 5. ALIGN 700-HZ LABELS TO 32 HZ
# ============================================================

def align_labels_by_time(labels, label_fs, target_fs, target_length):
    labels = np.asarray(labels).squeeze()
    if labels.ndim != 1:
        raise ValueError(f"Labels must be 1-D, got {labels.shape}")
    if len(labels) == 0:
        raise ValueError("Labels array is empty.")

    target_times = np.arange(target_length, dtype=np.float64) / float(target_fs)
    indices = np.floor(target_times * label_fs).astype(np.int64)
    indices = np.clip(indices, 0, len(labels) - 1)
    return labels[indices]


# ============================================================
# 6. HRV FEATURES FROM ORIGINAL 64-HZ BVP
# ============================================================

def extract_hrv_features(bvp_window, fs):
    """
    Extract Mean RR, SDNN and RMSSD.

    This function now receives the original 64-Hz BVP so that
    peak detection is not performed after 64 -> 32 Hz reduction.
    """
    try:
        bvp_window = np.asarray(
            bvp_window, dtype=np.float32
        ).squeeze()

        if len(bvp_window) < 3:
            return np.zeros(3, dtype=np.float32)

        if not np.all(np.isfinite(bvp_window)):
            finite_mask = np.isfinite(bvp_window)
            if finite_mask.sum() < 3:
                return np.zeros(3, dtype=np.float32)

            bvp_window = np.interp(
                np.arange(len(bvp_window)),
                np.flatnonzero(finite_mask),
                bvp_window[finite_mask]
            )

        min_distance = max(1, int(0.4 * fs))
        signal_std = float(np.std(bvp_window))

        if signal_std > 0:
            peaks, _ = find_peaks(
                bvp_window,
                distance=min_distance,
                prominence=0.10 * signal_std
            )
        else:
            peaks, _ = find_peaks(
                bvp_window,
                distance=min_distance
            )

        if len(peaks) < 3:
            return np.zeros(3, dtype=np.float32)

        rr = np.diff(peaks).astype(np.float64) / float(fs)
        rr = rr[(rr >= 0.3) & (rr <= 2.0)]

        if len(rr) < 2:
            return np.zeros(3, dtype=np.float32)

        mean_rr = np.mean(rr)
        sdnn = np.std(rr)
        diff_rr = np.diff(rr)
        rmssd = np.sqrt(np.mean(diff_rr ** 2)) if len(diff_rr) else 0.0

        features = np.array(
            [mean_rr, sdnn, rmssd], dtype=np.float32
        )

        if not np.all(np.isfinite(features)):
            return np.zeros(3, dtype=np.float32)

        return features

    except Exception:
        return np.zeros(3, dtype=np.float32)


# ============================================================
# 7. CREATE WINDOWS
# ============================================================

def create_windows_with_features(
    eda,
    bvp_32,
    bvp_64,
    labels,
    fs_32,
    fs_64,
    window_size_s=60,
    step_size_s=10
):
    """
    Sequence input:
        EDA 32 Hz + BVP 32 Hz

    HRV input:
        Original BVP 64 Hz

    The 32-Hz and 64-Hz windows represent exactly the same
    60-second time intervals.
    """
    eda = np.asarray(eda, dtype=np.float32).squeeze()
    bvp_32 = np.asarray(bvp_32, dtype=np.float32).squeeze()
    bvp_64 = np.asarray(bvp_64, dtype=np.float32).squeeze()
    labels = np.asarray(labels).squeeze()

    duration_seconds = min(
        len(eda) / float(fs_32),
        len(bvp_32) / float(fs_32),
        len(bvp_64) / float(fs_64),
        len(labels) / float(fs_32)
    )

    common_length_32 = int(np.floor(duration_seconds * fs_32))
    common_length_64 = int(np.floor(duration_seconds * fs_64))

    eda = eda[:common_length_32]
    bvp_32 = bvp_32[:common_length_32]
    labels = labels[:common_length_32]
    bvp_64 = bvp_64[:common_length_64]

    window_samples_32 = int(window_size_s * fs_32)
    step_samples_32 = int(step_size_s * fs_32)
    window_samples_64 = int(window_size_s * fs_64)

    if common_length_32 < window_samples_32:
        return (
            np.empty((0, window_samples_32, 2), dtype=np.float32),
            np.empty((0, 3), dtype=np.float32),
            np.empty((0,), dtype=np.int64)
        )

    sequences = []
    hrv_features = []
    window_labels = []

    for start_32 in range(
        0,
        common_length_32 - window_samples_32 + 1,
        step_samples_32
    ):
        end_32 = start_32 + window_samples_32

        start_time = start_32 / float(fs_32)
        end_time = end_32 / float(fs_32)

        start_64 = int(round(start_time * fs_64))
        end_64 = int(round(end_time * fs_64))
        end_64 = min(end_64, len(bvp_64))

        eda_win = eda[start_32:end_32]
        bvp32_win = bvp_32[start_32:end_32]
        label_win = labels[start_32:end_32]
        bvp64_win = bvp_64[start_64:end_64]

        if len(bvp64_win) < window_samples_64:
            continue

        valid_labels = label_win[np.isin(label_win, [1, 2, 3])]
        if len(valid_labels) == 0:
            continue

        values, counts = np.unique(
            valid_labels, return_counts=True
        )
        majority_label = int(values[np.argmax(counts)])

        sequence = np.column_stack(
            [eda_win, bvp32_win]
        ).astype(np.float32)

        hrv = extract_hrv_features(
            bvp64_win, fs_64
        )

        sequences.append(sequence)
        hrv_features.append(hrv)
        window_labels.append(majority_label - 1)

    if len(sequences) == 0:
        return (
            np.empty((0, window_samples_32, 2), dtype=np.float32),
            np.empty((0, 3), dtype=np.float32),
            np.empty((0,), dtype=np.int64)
        )

    return (
        np.stack(sequences).astype(np.float32),
        np.stack(hrv_features).astype(np.float32),
        np.asarray(window_labels, dtype=np.int64)
    )


# ============================================================
# 8. CNN + POSITIONAL ENCODING + TRANSFORMER + HRV
# ============================================================

class StressClassifier(nn.Module):
    """
    Input:
        (B, 1920, 2)

    CNN:
        1920 -> 960 -> 480 -> 240
        2 -> 32 -> 64 -> 128 channels

    Transformer:
        (B, 240, 128)

    Then:
        Global Average Pooling -> 128
        + HRV -> 131
        Dense 64 -> 3 classes
    """

    def __init__(self, seq_channels=2, hrv_dim=3, n_classes=3):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(seq_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(CNN_DROPOUT),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(CNN_DROPOUT),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(CNN_DROPOUT)
        )

        # ----------------------------------------------------
        # OPTIMIZATION 1: Learnable positional encoding
        # ----------------------------------------------------
        self.positional_encoding = nn.Parameter(
            torch.empty(1, MAX_TRANSFORMER_LENGTH, 128)
        )
        nn.init.normal_(
            self.positional_encoding,
            mean=0.0,
            std=0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=TRANSFORMER_HEADS,
            dim_feedforward=256,
            dropout=TRANSFORMER_DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=TRANSFORMER_LAYERS
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(128 + hrv_dim, 64),
            nn.ReLU(),
            nn.Dropout(CLASSIFIER_DROPOUT),
            nn.Linear(64, n_classes)
        )

    def forward(self, x_seq, x_hrv):
        # [B, time, channels] -> [B, channels, time]
        x = x_seq.permute(0, 2, 1)
        x = self.cnn(x)

        # [B, 128, 240] -> [B, 240, 128]
        x = x.permute(0, 2, 1)

        sequence_length = x.size(1)
        if sequence_length > MAX_TRANSFORMER_LENGTH:
            raise ValueError(
                "Transformer sequence is longer than positional encoding."
            )

        # Positional information before Transformer.
        x = x + self.positional_encoding[:, :sequence_length, :]

        x = self.transformer(x)

        # [B, 240, 128] -> [B, 128, 240]
        x = x.permute(0, 2, 1)
        x = self.global_pool(x).squeeze(-1)

        # 128 CNN/Transformer features + 3 HRV features = 131
        x = torch.cat([x, x_hrv], dim=1)

        return self.classifier(x)


# ============================================================
# 9. WESAD PIPELINE
# ============================================================

class WESADPipeline:

    def __init__(self, dataset_path):
        self.dataset_path = dataset_path
        self.fs = TARGET_SAMPLING_RATE

    def load_subject_raw(self, subject):
        file_path = find_subject_file(self.dataset_path, subject)

        if file_path is None:
            raise FileNotFoundError(
                f"\n{subject}.pkl not found under:\n{self.dataset_path}"
            )

        print(f"\nLoading {subject}")
        print(f"File: {file_path}")

        with open(file_path, "rb") as f:
            data = pickle.load(f, encoding="latin1")

        if "signal" not in data:
            raise KeyError(f"{subject}: 'signal' key not found.")
        if "wrist" not in data["signal"]:
            raise KeyError(f"{subject}: wrist signal not found.")

        wrist = data["signal"]["wrist"]

        if "EDA" not in wrist:
            raise KeyError(f"{subject}: wrist EDA not found.")
        if "BVP" not in wrist:
            raise KeyError(f"{subject}: wrist BVP not found.")
        if "label" not in data:
            raise KeyError(f"{subject}: label not found.")

        eda_raw = np.asarray(
            wrist["EDA"], dtype=np.float32
        ).squeeze()
        bvp_raw = np.asarray(
            wrist["BVP"], dtype=np.float32
        ).squeeze()
        labels = np.asarray(data["label"]).squeeze()

        # EDA: 4 -> 32 Hz for CNN.
        eda = resample_signal(
            eda_raw, 4, self.fs
        )

        # BVP: keep original 64 Hz for HRV.
        bvp_64 = bvp_raw.astype(np.float32)

        # BVP: 64 -> 32 Hz for CNN.
        bvp_32 = resample_signal(
            bvp_raw, BVP_NATIVE_RATE, self.fs
        )

        common_length = min(
            len(eda), len(bvp_32)
        )

        eda = eda[:common_length]
        bvp_32 = bvp_32[:common_length]

        # Labels: 700 -> 32 Hz.
        labels = align_labels_by_time(
            labels,
            LABEL_NATIVE_RATE,
            self.fs,
            common_length
        )

        return eda, bvp_32, bvp_64, labels

    def build_all_subject_windows(self):
        all_sequences = []
        all_hrv = []
        all_labels = []
        all_groups = []

        for subject in SUBJECTS:
            print("\n" + "=" * 70)
            print(f"PROCESSING {subject}")
            print("=" * 70)

            eda, bvp_32, bvp_64, labels = self.load_subject_raw(subject)

            seq, hrv, y = create_windows_with_features(
                eda,
                bvp_32,
                bvp_64,
                labels,
                TARGET_SAMPLING_RATE,
                BVP_NATIVE_RATE,
                WINDOW_SIZE,
                STEP_SIZE
            )

            if len(y) == 0:
                print(f"WARNING: No valid windows found for {subject}.")
                continue

            all_sequences.append(seq)
            all_hrv.append(hrv)
            all_labels.append(y)
            all_groups.extend([subject] * len(y))

            print(f"Windows: {len(y)}")
            print(f"Sequence shape: {seq.shape}")
            print(f"HRV shape: {hrv.shape}")

        if len(all_sequences) == 0:
            raise RuntimeError("No windows were created from any subject.")

        return (
            np.concatenate(all_sequences, axis=0),
            np.concatenate(all_hrv, axis=0),
            np.concatenate(all_labels, axis=0),
            np.asarray(all_groups)
        )


# ============================================================
# 10. TRAIN ONE LOSO FOLD
# ============================================================

def train_one_fold(
    X_train_seq,
    X_test_seq,
    X_train_hrv,
    X_test_hrv,
    y_train,
    y_test
):
    # --------------------------------------------------------
    # Scaling: training data only
    # --------------------------------------------------------
    seq_scaler = StandardScaler()

    X_train_seq = seq_scaler.fit_transform(
        X_train_seq.reshape(-1, X_train_seq.shape[-1])
    ).reshape(X_train_seq.shape)

    X_test_seq = seq_scaler.transform(
        X_test_seq.reshape(-1, X_test_seq.shape[-1])
    ).reshape(X_test_seq.shape)

    hrv_scaler = StandardScaler()

    X_train_hrv = hrv_scaler.fit_transform(X_train_hrv)
    X_test_hrv = hrv_scaler.transform(X_test_hrv)

    # --------------------------------------------------------
    # Tensors
    # --------------------------------------------------------
    X_train_seq = torch.tensor(X_train_seq, dtype=torch.float32)
    X_test_seq = torch.tensor(X_test_seq, dtype=torch.float32)
    X_train_hrv = torch.tensor(X_train_hrv, dtype=torch.float32)
    X_test_hrv = torch.tensor(X_test_hrv, dtype=torch.float32)

    y_train_tensor = torch.tensor(y_train, dtype=torch.long)

    train_dataset = TensorDataset(
        X_train_seq,
        X_train_hrv,
        y_train_tensor
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    # --------------------------------------------------------
    # Class weights
    # --------------------------------------------------------
    classes = np.unique(y_train)
    class_weights = np.ones(3, dtype=np.float32)

    calculated_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    class_weights[classes] = calculated_weights

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------
    model = StressClassifier(
        seq_channels=2,
        hrv_dim=3,
        n_classes=3
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # --------------------------------------------------------
    # OPTIMIZATION 2: AdamW + Weight Decay
    # --------------------------------------------------------
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # --------------------------------------------------------
    # OPTIMIZATION 3: Learning-rate scheduler
    # --------------------------------------------------------
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        min_lr=MIN_LEARNING_RATE
    )

    # --------------------------------------------------------
    # OPTIMIZATION 4: Early stopping
    # --------------------------------------------------------
    best_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        num_batches = 0

        for batch_seq, batch_hrv, batch_y in train_loader:
            batch_seq = batch_seq.to(DEVICE)
            batch_hrv = batch_hrv.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(batch_seq, batch_hrv)
            loss = criterion(outputs, batch_y)
            loss.backward()

            optimizer.step()

            running_loss += loss.item()
            num_batches += 1

        avg_loss = running_loss / max(num_batches, 1)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:02d}/{EPOCHS} "
            f"| Loss: {avg_loss:.4f} "
            f"| LR: {current_lr:.7f}"
        )

        scheduler.step(avg_loss)

        if best_loss - avg_loss > EARLY_STOPPING_MIN_DELTA:
            best_loss = avg_loss
            epochs_without_improvement = 0

            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    # --------------------------------------------------------
    # Evaluation on held-out subject
    # --------------------------------------------------------
    model.eval()

    with torch.no_grad():
        outputs = model(
            X_test_seq.to(DEVICE),
            X_test_hrv.to(DEVICE)
        )
        predictions = torch.argmax(
            outputs, dim=1
        ).cpu().numpy()

    accuracy = accuracy_score(y_test, predictions)
    return accuracy, y_test, predictions


# ============================================================
# 11. LOSO VALIDATION
# ============================================================

def run_loso(X_seq, X_hrv, y, groups):
    logo = LeaveOneGroupOut()
    fold_accuracies = []
    fold_reports = []

    for fold, (train_idx, test_idx) in enumerate(
        logo.split(X_seq, y, groups), start=1
    ):
        test_subject = groups[test_idx][0]

        print("\n" + "=" * 70)
        print(f"LOSO FOLD {fold}")
        print(f"Test Subject: {test_subject}")
        print("=" * 70)

        X_train_seq = X_seq[train_idx]
        X_test_seq = X_seq[test_idx]
        X_train_hrv = X_hrv[train_idx]
        X_test_hrv = X_hrv[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        print(f"Train windows: {len(y_train)}")
        print(f"Test windows: {len(y_test)}")

        accuracy, y_true, predictions = train_one_fold(
            X_train_seq,
            X_test_seq,
            X_train_hrv,
            X_test_hrv,
            y_train,
            y_test
        )

        fold_accuracies.append(accuracy)

        print(f"\nFold Accuracy: {accuracy * 100:.2f}%")

        report = classification_report(
            y_true,
            predictions,
            labels=[0, 1, 2],
            target_names=["Baseline", "Stress", "Amusement"],
            zero_division=0
        )

        fold_reports.append(report)

        print("\nClassification Report:")
        print(report)

    fold_accuracies = np.asarray(
        fold_accuracies, dtype=np.float64
    )

    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)

    print("\n" + "=" * 70)
    print("FINAL LOSO RESULTS")
    print("=" * 70)

    for i, acc in enumerate(fold_accuracies, start=1):
        print(f"Fold {i:02d}: {acc * 100:.2f}%")

    print("-" * 70)
    print(f"Mean Accuracy: {mean_accuracy * 100:.2f}%")
    print(f"Std Accuracy: {std_accuracy * 100:.2f}%")
    print("=" * 70)

    return fold_accuracies, fold_reports


# ============================================================
# 12. MAIN
# ============================================================

def main():
    print("\n" + "=" * 70)
    print("WESAD STRESS DETECTION - OPTIMIZED PIPELINE")
    print("=" * 70)
    print(f"Device: {DEVICE}")
    print(f"Target Sampling Rate: {TARGET_SAMPLING_RATE} Hz")
    print(f"Window: {WINDOW_SIZE} seconds")
    print(f"Step: {STEP_SIZE} seconds")
    print("Input: EDA + BVP")
    print("HRV: Mean RR, SDNN, RMSSD from original 64-Hz BVP")
    print(
        "Model: 3-Layer CNN + BatchNorm + Dropout "
        "+ Positional Encoding + Transformer + HRV"
    )
    print(
        "Optimization: AdamW + Weight Decay + "
        "LR Scheduler + Early Stopping"
    )
    print("Classes: Baseline, Stress, Amusement")

    dataset_path = download_wesad_dataset()

    pipeline = WESADPipeline(dataset_path)
    X_seq, X_hrv, y, groups = pipeline.build_all_subject_windows()

    print("\n" + "=" * 70)
    print("DATASET SUMMARY")
    print("=" * 70)
    print(f"X_seq shape : {X_seq.shape}")
    print(f"X_hrv shape : {X_hrv.shape}")
    print(f"y shape     : {y.shape}")
    print(f"groups shape: {groups.shape}")

    class_names = ["Baseline", "Stress", "Amusement"]
    print("\nClass distribution:")
    for class_id, class_name in enumerate(class_names):
        count = int(np.sum(y == class_id))
        print(f"{class_name}: {count}")

    print("\nSubjects found:")
    print(np.unique(groups))

    return run_loso(X_seq, X_hrv, y, groups)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    fold_accuracies, fold_reports = main()


MOUNTING GOOGLE DRIVE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Google Drive mounted.
WESAD storage location:
/content/drive/MyDrive/WESAD

WESAD STRESS DETECTION - OPTIMIZED PIPELINE
Device: cuda
Target Sampling Rate: 32 Hz
Window: 60 seconds
Step: 10 seconds
Input: EDA + BVP
HRV: Mean RR, SDNN, RMSSD from original 64-Hz BVP
Model: 3-Layer CNN + BatchNorm + Dropout + Positional Encoding + Transformer + HRV
Optimization: AdamW + Weight Decay + LR Scheduler + Early Stopping
Classes: Baseline, Stress, Amusement
WESAD DATASET SETUP

WESAD dataset already exists in Google Drive.
No download required.
No extraction required.

S2.pkl found:
/content/drive/MyDrive/WESAD/WESAD/S2/S2.pkl

Dataset root:
/content/drive/MyDrive/WESAD/WESAD

PROCESSING S2

Loading S2
File: /content/drive/MyDrive/WESAD/WESAD/S2/S2.pkl
Windows: 230
Sequence shape: (230, 1920, 2)
HRV shape: (230, 3)

PROCESSING S3

Loading S3
File:

/tmp/ipykernel_2906/1938445514.py:504: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch 01/30 | Loss: 0.8988 | LR: 0.0010000
Epoch 02/30 | Loss: 0.6820 | LR: 0.0010000
Epoch 03/30 | Loss: 0.6039 | LR: 0.0010000
Epoch 04/30 | Loss: 0.4632 | LR: 0.0010000
Epoch 05/30 | Loss: 0.4836 | LR: 0.0010000
Epoch 06/30 | Loss: 0.4204 | LR: 0.0010000
Epoch 07/30 | Loss: 0.4098 | LR: 0.0010000
Epoch 08/30 | Loss: 0.3477 | LR: 0.0010000
Epoch 09/30 | Loss: 0.3202 | LR: 0.0010000
Epoch 10/30 | Loss: 0.3154 | LR: 0.0010000
Epoch 11/30 | Loss: 0.3047 | LR: 0.0010000
Epoch 12/30 | Loss: 0.2815 | LR: 0.0010000
Epoch 13/30 | Loss: 0.3179 | LR: 0.0010000
Epoch 14/30 | Loss: 0.2840 | LR: 0.0010000
Epoch 15/30 | Loss: 0.2783 | LR: 0.0010000
Epoch 16/30 | Loss: 0.2463 | LR: 0.0010000
Epoch 17/30 | Loss: 0.2820 | LR: 0.0010000
Epoch 18/30 | Loss: 0.2145 | LR: 0.0010000
Epoch 19/30 | Loss: 0.2091 | LR: 0.0010000
Epoch 20/30 | Loss: 0.2307 | LR: 0.0010000
Epoch 21/30 | Loss: 0.2305 | LR: 0.0010000
Epoch 22/30 | Loss: 0.1917 | LR: 0.0010000
Epoch 23/30 | Loss: 0.1827 | LR: 0.0010000
Epoch 24/30

/tmp/ipykernel_2906/1938445514.py:504: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch 01/30 | Loss: 0.9107 | LR: 0.0010000
Epoch 02/30 | Loss: 0.7756 | LR: 0.0010000
Epoch 03/30 | Loss: 0.6199 | LR: 0.0010000
Epoch 04/30 | Loss: 0.5708 | LR: 0.0010000
Epoch 05/30 | Loss: 0.4783 | LR: 0.0010000
Epoch 06/30 | Loss: 0.4548 | LR: 0.0010000
Epoch 07/30 | Loss: 0.4050 | LR: 0.0010000
Epoch 08/30 | Loss: 0.3780 | LR: 0.0010000
Epoch 09/30 | Loss: 0.3848 | LR: 0.0010000
Epoch 10/30 | Loss: 0.3210 | LR: 0.0010000
Epoch 11/30 | Loss: 0.3442 | LR: 0.0010000
Epoch 12/30 | Loss: 0.2978 | LR: 0.0010000
Epoch 13/30 | Loss: 0.3509 | LR: 0.0010000
Epoch 14/30 | Loss: 0.2902 | LR: 0.0010000
Epoch 15/30 | Loss: 0.2759 | LR: 0.0010000
Epoch 16/30 | Loss: 0.2528 | LR: 0.0010000
Epoch 17/30 | Loss: 0.2347 | LR: 0.0010000
Epoch 18/30 | Loss: 0.2478 | LR: 0.0010000
Epoch 19/30 | Loss: 0.1903 | LR: 0.0010000
Epoch 20/30 | Loss: 0.2047 | LR: 0.0010000
Epoch 21/30 | Loss: 0.2136 | LR: 0.0010000
Epoch 22/30 | Loss: 0.1782 | LR: 0.0010000
Epoch 23/30 | Loss: 0.1562 | LR: 0.0010000
Epoch 24/30

...

Epoch 01/30 | Loss: 0.9245 | LR: 0.0010000
Epoch 02/30 | Loss: 0.7262 | LR: 0.0010000
Epoch 03/30 | Loss: 0.5869 | LR: 0.0010000
Epoch 04/30 | Loss: 0.4870 | LR: 0.0010000
Epoch 05/30 | Loss: 0.4328 | LR: 0.0010000
Epoch 06/30 | Loss: 0.4107 | LR: 0.0010000
Epoch 07/30 | Loss: 0.3741 | LR: 0.0010000
Epoch 08/30 | Loss: 0.3671 | LR: 0.0010000
Epoch 09/30 | Loss: 0.2976 | LR: 0.0010000
Epoch 10/30 | Loss: 0.3135 | LR: 0.0010000
Epoch 11/30 | Loss: 0.2909 | LR: 0.0010000
Epoch 12/30 | Loss: 0.2712 | LR: 0.0010000
Epoch 13/30 | Loss: 0.2412 | LR: 0.0010000
Epoch 14/30 | Loss: 0.2241 | LR: 0.0010000
Epoch 15/30 | Loss: 0.2348 | LR: 0.0010000
Epoch 16/30 | Loss: 0.2301 | LR: 0.0010000
Epoch 17/30 | Loss: 0.1995 | LR: 0.0010000
Epoch 18/30 | Loss: 0.2060 | LR: 0.0010000
Epoch 19/30 | Loss: 0.2434 | LR: 0.0010000
Epoch 20/30 | Loss: 0.1956 | LR: 0.0010000
Epoch 21/30 | Loss: 0.1629 | LR: 0.0010000
Epoch 22/30 | Loss: 0.1720 | LR: 0.0010000
Epoch 23/30 | Loss: 0.1441 | LR: 0.0010000
Epoch 24/30

/tmp/ipykernel_2906/1938445514.py:504: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch 01/30 | Loss: 0.9287 | LR: 0.0010000
Epoch 02/30 | Loss: 0.7321 | LR: 0.0010000
Epoch 03/30 | Loss: 0.5678 | LR: 0.0010000
Epoch 04/30 | Loss: 0.4994 | LR: 0.0010000
Epoch 05/30 | Loss: 0.4230 | LR: 0.0010000
Epoch 06/30 | Loss: 0.4248 | LR: 0.0010000
Epoch 07/30 | Loss: 0.3803 | LR: 0.0010000
Epoch 08/30 | Loss: 0.3705 | LR: 0.0010000
Epoch 09/30 | Loss: 0.3208 | LR: 0.0010000
Epoch 10/30 | Loss: 0.3139 | LR: 0.0010000
Epoch 11/30 | Loss: 0.3122 | LR: 0.0010000
Epoch 12/30 | Loss: 0.2750 | LR: 0.0010000
Epoch 13/30 | Loss: 0.2763 | LR: 0.0010000
Epoch 14/30 | Loss: 0.2751 | LR: 0.0010000
Epoch 15/30 | Loss: 0.2359 | LR: 0.0010000
Epoch 16/30 | Loss: 0.2150 | LR: 0.0010000
Epoch 17/30 | Loss: 0.2028 | LR: 0.0010000
Epoch 18/30 | Loss: 0.1986 | LR: 0.0010000
Epoch 19/30 | Loss: 0.1905 | LR: 0.0010000
Epoch 20/30 | Loss: 0.1835 | LR: 0.0010000
Epoch 21/30 | Loss: 0.2084 | LR: 0.0010000
Epoch 22/30 | Loss: 0.1620 | LR: 0.0010000
Epoch 23/30 | Loss: 0.1314 | LR: 0.0010000
Epoch 24/30

/tmp/ipykernel_2906/1938445514.py:504: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch 01/30 | Loss: 0.9468 | LR: 0.0010000
Epoch 02/30 | Loss: 0.7811 | LR: 0.0010000
Epoch 03/30 | Loss: 0.6114 | LR: 0.0010000
Epoch 04/30 | Loss: 0.5076 | LR: 0.0010000
Epoch 05/30 | Loss: 0.4749 | LR: 0.0010000
Epoch 06/30 | Loss: 0.4250 | LR: 0.0010000
Epoch 07/30 | Loss: 0.3955 | LR: 0.0010000
Epoch 08/30 | Loss: 0.3645 | LR: 0.0010000
Epoch 09/30 | Loss: 0.3594 | LR: 0.0010000
Epoch 10/30 | Loss: 0.3487 | LR: 0.0010000
Epoch 11/30 | Loss: 0.3200 | LR: 0.0010000
Epoch 12/30 | Loss: 0.2840 | LR: 0.0010000
Epoch 13/30 | Loss: 0.2873 | LR: 0.0010000
Epoch 14/30 | Loss: 0.2434 | LR: 0.0010000
Epoch 15/30 | Loss: 0.2482 | LR: 0.0010000
Epoch 16/30 | Loss: 0.2631 | LR: 0.0010000
Epoch 17/30 | Loss: 0.2141 | LR: 0.0010000
Epoch 18/30 | Loss: 0.1816 | LR: 0.0010000
Epoch 19/30 | Loss: 0.1954 | LR: 0.0010000
Epoch 20/30 | Loss: 0.1829 | LR: 0.0010000
Epoch 21/30 | Loss: 0.1858 | LR: 0.0010000
Epoch 22/30 | Loss: 0.1298 | LR: 0.0005000
Epoch 23/30 | Loss: 0.1156 | LR: 0.0005000
Epoch 24/30